In [1]:
import math
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# =============== 1) ELO ===============
K, HFA, INICIO = 40, 100, "1995-01-01"
INI_OTROS = 900
TOP_INICIAL = {"RSA","NZL","IRE","FRA","ENG","SCO","AUS","ARG","WAL","ITA"}

df = pd.read_csv("matches_actualizado.csv", parse_dates=["date"])
d = df.dropna(subset=["home_score", "away_score"])
d = d[d["date"] >= INICIO].sort_values("date", kind="stable").reset_index(drop=True)

elo, filas = {}, []
def rating(t):
    if t not in elo:
        elo[t] = 1500.0 if t in TOP_INICIAL else float(INI_OTROS)
    return elo[t]

for r in d.itertuples():
    eh, ea = rating(r.home), rating(r.away)
    ventaja = HFA * r.venue_country
    p_home = 1 / (1 + 10 ** (-(eh + ventaja - ea) / 400))
    dif = r.home_score - r.away_score
    real = 1.0 if dif > 0 else (0.0 if dif < 0 else 0.5)
    gd = (eh + ventaja - ea) if dif > 0 else (ea - eh - ventaja)
    mult = math.log(abs(dif) + 1) * 2.2 / (gd * 0.001 + 2.2) if dif != 0 else 1.0
    c = K * mult * (real - p_home)
    elo[r.home], elo[r.away] = eh + c, ea - c
    filas.append((eh, ea, ventaja, p_home))

res = d[["date", "home", "away", "home_score", "away_score", "comp", "venue_country"]].copy()
res[["elo_home", "elo_away", "ventaja", "p_elo"]] = filas
res["ganó_local"] = np.where(res.home_score > res.away_score, 1.0,
                     np.where(res.home_score < res.away_score, 0.0, np.nan))

# =============== 2) VARIABLES (solo con partidos anteriores) ===============
N = 5   # ventana de forma reciente
largo = pd.concat([
    pd.DataFrame({"idx": res.index, "lado": "home", "eq": res.home, "date": res.date,
                  "gano": (res.home_score > res.away_score).astype(float),
                  "margen": res.home_score - res.away_score}),
    pd.DataFrame({"idx": res.index, "lado": "away", "eq": res.away, "date": res.date,
                  "gano": (res.away_score > res.home_score).astype(float),
                  "margen": res.away_score - res.home_score}),
]).sort_values(["eq", "date", "idx"], kind="stable")

g = largo.groupby("eq")
largo["forma_gan"]    = g["gano"].transform(lambda s: s.shift(1).rolling(N, min_periods=1).mean())
largo["forma_margen"] = g["margen"].transform(lambda s: s.shift(1).rolling(N, min_periods=1).mean())
largo["descanso"]     = g["date"].transform(lambda s: s.diff().dt.days)

for lado in ["home", "away"]:
    t = largo[largo.lado == lado].set_index("idx")[["forma_gan", "forma_margen", "descanso"]]
    t.columns = [f"{c}_{lado}" for c in t.columns]
    res = res.join(t)

res["elo_diff"] = res.elo_home + res.ventaja - res.elo_away
res["neutral"] = (res.venue_country == 0).astype(int)
res["dif_forma_gan"] = res.forma_gan_home - res.forma_gan_away
res["dif_forma_margen"] = res.forma_margen_home - res.forma_margen_away
res["descanso_home"] = res.descanso_home.clip(upper=180).fillna(180)
res["descanso_away"] = res.descanso_away.clip(upper=180).fillna(180)
res["dif_descanso"] = res.descanso_home - res.descanso_away
res[["dif_forma_gan", "dif_forma_margen"]] = res[["dif_forma_gan", "dif_forma_margen"]].fillna(0)
for c in ["rwc", "rwc_qual", "six_nations", "rugby_championship", "tier2_comp", "warmup"]:
    res[f"comp_{c}"] = (res.comp == c).astype(int)

# =============== 3) PESOS ===============
MUNDIAL = {"FRA","NZL","ITA","IRE","RSA","SCO","WAL","FIJ","AUS","ENG","ARG","JPN",
           "URU","USA","ESP","GEO","POR","ROU","CHI","TON","SAM","HKG","CAN","ZIM"}
PESO_DOS, PESO_UNO, PESO_NINGUNO = 1.0, 0.75, 0.5   # equipos del Mundial en el partido: ambos / uno / ninguno

res["n_mundial"] = res.home.isin(MUNDIAL).astype(int) + res.away.isin(MUNDIAL).astype(int)
res["peso"] = res.n_mundial.map({2: PESO_DOS, 1: PESO_UNO, 0: PESO_NINGUNO})

datos = res[(res.date >= "2015-01-01") & res.ganó_local.notna()].copy()

VARS = ["elo_diff", "neutral", "dif_forma_gan", "dif_forma_margen", "dif_descanso",
        "comp_rwc", "comp_rwc_qual", "comp_six_nations", "comp_rugby_championship",
        "comp_tier2_comp", "comp_warmup"]

def logloss(y, p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return -np.mean(np.where(y == 1, np.log(p), np.log(1 - p)))

# =============== 4) EVALUACIÓN EN 4 BLOQUES (entrena con todo lo anterior) ===============
# Se evalúa siempre sobre partidos donde juega al menos una selección del Mundial.
bloques = [("2019-01-01", "2021-01-01"), ("2021-01-01", "2023-01-01"),
           ("2023-01-01", "2025-01-01"), ("2025-01-01", "2027-01-01")]
filas = []
for a, b in bloques:
    tr = datos[datos.date < a]
    te = datos[(datos.date >= a) & (datos.date < b) & (datos.n_mundial >= 1)]
    m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
    m.fit(tr[VARS], tr.ganó_local, logisticregression__sample_weight=tr.peso.values)
    p = m.predict_proba(te[VARS])[:, 1]
    filas.append((f"{a[:4]}-{b[:4]}", len(tr), len(te),
                  logloss(te.ganó_local.values, te.p_elo.values),
                  logloss(te.ganó_local.values, p),
                  np.mean((p > 0.5) == (te.ganó_local.values == 1))))
tabla = pd.DataFrame(filas, columns=["bloque", "n_train", "n_test", "logloss_Elo", "logloss_modelo", "acierto_modelo"])
print(tabla.round(4).to_string(index=False))
w = tabla.n_test
print("\nPromedio ponderado -> Elo:", round((tabla.logloss_Elo * w).sum() / w.sum(), 4),
      "| modelo:", round((tabla.logloss_modelo * w).sum() / w.sum(), 4))

   bloque  n_train  n_test  logloss_Elo  logloss_modelo  acierto_modelo
2019-2021      825     182       0.4134          0.4320          0.8077
2021-2023     1069     215       0.5246          0.5179          0.7302
2023-2025     1386     245       0.4053          0.3985          0.8327
2025-2027     1769     223       0.4527          0.4350          0.7937

Promedio ponderado -> Elo: 0.4489 | modelo: 0.4446


In [2]:
# =============== CELDA 2: MODELO DEL MARCADOR (corre después de la celda 1) ===============
from scipy.stats import norm

# ---- modelo de probabilidad final, entrenado con todos los datos ----
# Se promedian dos modelos: uno completo y otro SIN la forma reciente en victorias
# (esa variable mejora la validación pero tiene un signo contraintuitivo; el promedio es un punto medio).
VARS_SIN = [v for v in VARS if v != "dif_forma_gan"]
def _ajustar(cols):
    m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
    m.fit(datos[cols], datos.ganó_local, logisticregression__sample_weight=datos.peso.values)
    return m
_mod_a, _mod_b = _ajustar(VARS), _ajustar(VARS_SIN)

class Promedio:
    """Se usa igual que un modelo de sklearn: predict_proba(DataFrame con todas las variables)."""
    def predict_proba(self, X):
        p = (_mod_a.predict_proba(X[VARS])[:, 1] + _mod_b.predict_proba(X[VARS_SIN])[:, 1]) / 2
        return np.column_stack([1 - p, p])
mod = Promedio()

# ---- historial desde 2015 (incluye empates) en "perspectiva del favorito" ----
hist = res[res.date >= "2015-01-01"].copy()
hist["home_tries"] = d.loc[hist.index, "home_tries"]; hist["away_tries"] = d.loc[hist.index, "away_tries"]
hist["p"] = mod.predict_proba(hist[VARS])[:, 1]          # P(gana el local), sin contar empates
hist["fav_local"] = hist.p >= 0.5
hist["p_fav"] = np.where(hist.fav_local, hist.p, 1 - hist.p)
hist["fz"] = norm.ppf(hist.p_fav.clip(0.5, 0.999))       # qué tan desparejo es el partido
hist["pts_fav"] = np.where(hist.fav_local, hist.home_score, hist.away_score)
hist["pts_dog"] = np.where(hist.fav_local, hist.away_score, hist.home_score)
hist["tries_fav"] = np.where(hist.fav_local, hist.home_tries, hist.away_tries)
hist["tries_dog"] = np.where(hist.fav_local, hist.away_tries, hist.home_tries)
hist["empate"] = hist.pts_fav == hist.pts_dog
hist["fav_gano"] = hist.pts_fav > hist.pts_dog

K_TRIES = 1.15     # ajuste de ensayos, calibrado con 2025 (1.0 = sin ajuste)
VIDA_MEDIA = 1.0   # años: un partido de hace 1 año pesa la mitad que uno de hoy (el rugby anota cada vez más)
def recientes(h, fecha_ref, vida_media=VIDA_MEDIA, n=6000, seed=0):
    """Remuestrea el historial dando más peso a los partidos recientes."""
    edad = (pd.Timestamp(fecha_ref) - h.date).dt.days / 365.25
    w = 0.5 ** (edad / vida_media); w = w / w.sum()
    return h.loc[np.random.default_rng(seed).choice(h.index, n, p=w)]

class Marcador:
    """Simula puntos y tries de un partido a partir de la probabilidad de que gane el favorito."""
    def __init__(self, h, rng, k=K_TRIES):
        self.rng = rng
        self.k = k   # ajuste de ensayos: sortea con el historial de equipos de P*k puntos
        self.p_empate = h.empate.mean()
        nd = h[~h.empate]
        # margen del ganador: lognormal cuya media depende de lo desparejo del partido y de si ganó el favorito
        W = (nd.pts_fav - nd.pts_dog).abs()
        X = np.column_stack([np.ones(len(nd)), nd.fz, nd.fav_gano.astype(float)])
        self.beta, *_ = np.linalg.lstsq(X, np.log(W), rcond=None)
        self.sd_w = (np.log(W) - X @ self.beta).std()
        # puntos del perdedor dado el margen (muestreo empírico con margen parecido)
        perd = np.minimum(nd.pts_fav, nd.pts_dog).values
        self.W_hist, self.L_hist = W.values, perd
        self.L_por_W = {w: perd[np.abs(W.values - w) <= (2 if w < 40 else 8)] for w in range(1, 111)}
        self.puntos_empate = h[h.empate].pts_fav.values
        # tries dado los puntos (muestreo empírico)
        ht = h.dropna(subset=["tries_fav", "tries_dog"])
        self.pts_all = np.concatenate([ht.pts_fav, ht.pts_dog]); self.tr_all = np.concatenate([ht.tries_fav, ht.tries_dog])
        self.tr_por_p = {}
        for x in range(0, 141):
            v = 2
            while True:
                s = self.tr_all[np.abs(self.pts_all - x) <= v]
                if len(s) >= 20 or v > 25: break
                v += 1
            self.tr_por_p[x] = s

    def tries(self, pts):
        return self.rng.choice(self.tr_por_p[int(min(round(pts * self.k), 140))])

    def simular(self, p_fav):
        """Devuelve (pts_fav, pts_dog, tries_fav, tries_dog)."""
        r = self.rng
        u = r.random()
        if u < self.p_empate:
            x = int(r.choice(self.puntos_empate)); return x, x, self.tries(x), self.tries(x)
        fav_gana = (r.random() < p_fav)
        fz = norm.ppf(min(max(p_fav, 0.5), 0.999))
        mu = self.beta[0] + self.beta[1] * fz + self.beta[2] * float(fav_gana)
        W = int(max(1, round(np.exp(mu + self.sd_w * r.standard_normal()))))
        L = int(r.choice(self.L_por_W[min(W, 110)]))
        G = L + W
        pf, pd_ = (G, L) if fav_gana else (L, G)
        return pf, pd_, self.tries(pf), self.tries(pd_)

def bonus_puntos(pts_a, pts_b, tries_a, tries_b):
    """Bonus de A: +1 por 4+ tries, +1 si pierde por menos de 8."""
    return int(tries_a >= 4) + int(pts_a < pts_b and pts_b - pts_a < 8)

# =============== VALIDACIÓN: ajustar con partidos hasta 2024 y probar en 2025-2026 ===============
rng = np.random.default_rng(123)
m_fit = Marcador(recientes(hist[hist.date < "2025-01-01"], "2025-01-01"), rng)
test2 = hist[hist.date >= "2025-01-01"]
N_SIM = 100

filas = []
for r_ in test2.itertuples():
    for _ in range(N_SIM):
        pf, pd_, tf, td = m_fit.simular(r_.p_fav)
        filas.append((r_.Index, r_.fz, pf, pd_, tf, td))
sim = pd.DataFrame(filas, columns=["idx", "fz", "pts_fav", "pts_dog", "tries_fav", "tries_dog"])
sim["M"] = (sim.pts_fav - sim.pts_dog).abs()

def resumen(x, tries_ok=True):
    M = (x.pts_fav - x.pts_dog).abs()
    out = {
        "empates": (M == 0).mean(),
        "margen < 8": (M < 8).mean(),
        "margen >= 30": (M >= 30).mean(),
        "puntos totales (media)": (x.pts_fav + x.pts_dog).mean(),
        "favorito gana": (x.pts_fav > x.pts_dog).mean(),
    }
    if tries_ok:
        t4 = np.concatenate([(x.tries_fav >= 4), (x.tries_dog >= 4)]).mean()
        lb = np.concatenate([(x.pts_fav < x.pts_dog) & (x.pts_dog - x.pts_fav < 8), (x.pts_dog < x.pts_fav) & (x.pts_fav - x.pts_dog < 8)])
        perdio = np.concatenate([x.pts_fav < x.pts_dog, x.pts_dog < x.pts_fav])
        out["P(4+ tries) por equipo"] = t4
        out["P(4+ tries) favorito"] = (x.tries_fav >= 4).mean()
        out["P(4+ tries) no favorito"] = (x.tries_dog >= 4).mean()
        out["P(bonus perdedor | perdió)"] = lb.sum() / perdio.sum()
        out["bonus por equipo y partido"] = t4 + lb.mean()
    return out

real_t = test2.dropna(subset=["tries_fav", "tries_dog"])
sim_t = sim[sim.idx.isin(real_t.index)]
comp = pd.DataFrame({"real": resumen(real_t), "simulado": resumen(sim_t)})
print(f"Partidos de prueba: {len(test2)} ({len(real_t)} con tries) | {N_SIM} simulaciones por partido\n")
print(comp.round(3).to_string())

# por nivel de desequilibrio del partido
print("\nMargen medio del ganador, real vs simulado, según qué tan desparejo es el partido:")
test2 = test2.assign(M=(test2.pts_fav - test2.pts_dog).abs(), gb=pd.cut(test2.fz, [-0.01, .5, 1, 1.5, 3]))
sim["gb"] = sim.idx.map(test2.gb)
t = pd.DataFrame({"real": test2[test2.M > 0].groupby("gb", observed=True).M.mean(),
                  "simulado": sim[sim.M > 0].groupby("gb", observed=True).M.mean(),
                  "n_real": test2.groupby("gb", observed=True).size()})
print(t.round(1).to_string())

Partidos de prueba: 354 (212 con tries) | 100 simulaciones por partido

                              real  simulado
empates                      0.024     0.012
margen < 8                   0.302     0.285
margen >= 30                 0.217     0.258
puntos totales (media)      57.495    56.628
favorito gana                0.764     0.762
P(4+ tries) por equipo       0.488     0.445
P(4+ tries) favorito         0.684     0.645
P(4+ tries) no favorito      0.292     0.246
P(bonus perdedor | perdió)   0.285     0.277
bonus por equipo y partido   0.627     0.582

Margen medio del ganador, real vs simulado, según qué tan desparejo es el partido:
              real  simulado  n_real
gb                                  
(-0.01, 0.5]  19.1      16.6     115
(0.5, 1.0]    20.5      23.0     108
(1.0, 1.5]    22.6      31.3      68
(1.5, 3.0]    42.5      42.8      63


In [3]:
# =============== CELDA 3: FASE DE GRUPOS (corre después de las celdas 1 y 2) ===============
import time
from collections import Counter

POOLS = {"A": ["AUS","NZL","CHI","HKG"], "B": ["RSA","ITA","GEO","ROU"],
         "C": ["FIJ","ESP","ARG","CAN"], "D": ["SCO","URU","IRE","POR"],
         "E": ["FRA","USA","JPN","SAM"], "F": ["ENG","WAL","TON","ZIM"]}
POOL_DE = {t: p for p, ts in POOLS.items() for t in ts}

# ---- 1) los 36 partidos de grupo con sus variables (foto de hoy) ----
fx = df[(df.comp == "rwc") & df.home_score.isna() & (df.date >= "2027-01-01")].sort_values("date").reset_index(drop=True)
assert len(fx) == 36 and all(POOL_DE[a] == POOL_DE[b] for a, b in zip(fx.home, fx.away)), "revisar fixtures/grupos"

ult = largo.groupby("eq").tail(N).groupby("eq").agg(forma_gan=("gano", "mean"), forma_margen=("margen", "mean"))
fx["elo_h"], fx["elo_a"] = fx.home.map(elo), fx.away.map(elo)
fx["ventaja"] = HFA * fx.venue_country
fx["elo_diff"] = fx.elo_h + fx.ventaja - fx.elo_a
fx["neutral"] = (fx.venue_country == 0).astype(int)
fx["dif_forma_gan"] = fx.home.map(ult.forma_gan) - fx.away.map(ult.forma_gan)
fx["dif_forma_margen"] = fx.home.map(ult.forma_margen) - fx.away.map(ult.forma_margen)
# descanso: antes de su primer partido, 180 (el Mundial queda lejos del último partido registrado);
# después, los días desde su partido anterior del Mundial
prev = {}; desc_h, desc_a = [], []
for r_ in fx.itertuples():
    desc_h.append(min((r_.date - prev[r_.home]).days, 180) if r_.home in prev else 180)
    desc_a.append(min((r_.date - prev[r_.away]).days, 180) if r_.away in prev else 180)
    prev[r_.home] = prev[r_.away] = r_.date
fx["dif_descanso"] = np.array(desc_h) - np.array(desc_a)
for c in ["rwc", "rwc_qual", "six_nations", "rugby_championship", "tier2_comp", "warmup"]:
    fx[f"comp_{c}"] = int(c == "rwc")
fx["p_local"] = mod.predict_proba(fx[VARS])[:, 1]       # P(gana el local), sin contar empates

# ---- 2) simulador de marcador ajustado con todo el historial (con más peso a lo reciente) ----
rng = np.random.default_rng(2027)
marc = Marcador(recientes(hist, hist.date.max()), rng)

# ---- 3) una fase de grupos completa ----
ELO_RANK = {t: elo[t] for t in elo}      # aproxima el ranking mundial para el último desempate
def fase_de_grupos():
    st = {t: dict(pts=0, dif=0, pf=0, tr=0) for t in POOL_DE}
    gano = {}                                            # gano[(a,b)] = True si a le ganó a b
    for r_ in fx.itertuples():
        p = r_.p_local; fav_local = p >= 0.5
        pf, pd_, tf, td = marc.simular(p if fav_local else 1 - p)
        ph, pa, th, ta = (pf, pd_, tf, td) if fav_local else (pd_, pf, td, tf)
        for eq, a_fav, en_c, tr in ((r_.home, ph, pa, th), (r_.away, pa, ph, ta)):
            s = st[eq]
            s["pts"] += (4 if a_fav > en_c else 2 if a_fav == en_c else 0) + int(tr >= 4) + int(a_fav < en_c and en_c - a_fav < 8)
            s["dif"] += a_fav - en_c; s["pf"] += a_fav; s["tr"] += tr
        gano[(r_.home, r_.away)] = ph > pa; gano[(r_.away, r_.home)] = pa > ph
    orden, terceros = {}, []
    for pool, eqs in POOLS.items():
        res_pool = []
        for pts in sorted({st[t]["pts"] for t in eqs}, reverse=True):
            g = [t for t in eqs if st[t]["pts"] == pts]
            if len(g) == 2 and (gano[(g[0], g[1])] or gano[(g[1], g[0])]):   # empate de 2: gana el que ganó el cruce
                g = g if gano[(g[0], g[1])] else g[::-1]
            else:
                g = sorted(g, key=lambda t: (st[t]["dif"], st[t]["pf"], st[t]["tr"], ELO_RANK[t]), reverse=True)
            res_pool += g
        orden[pool] = res_pool
        terceros.append(res_pool[2])
    # mejores 4 terceros: puntos, diferencia de puntos, puntos a favor, tries a favor, ranking
    terceros.sort(key=lambda t: (st[t]["pts"], st[t]["dif"], st[t]["pf"], st[t]["tr"], ELO_RANK[t]), reverse=True)
    return orden, terceros[:4], st

# ---- 4) 10.000 fases de grupos: ¿quién avanza? ----
N_TORNEOS = 10_000
pos = {t: np.zeros(4) for t in POOL_DE}; como_3ro = Counter(); combos = Counter(); pts_tot = Counter()
t0 = time.time()
for _ in range(N_TORNEOS):
    orden, mejores3, st = fase_de_grupos()
    for pool, lst in orden.items():
        for i, t in enumerate(lst): pos[t][i] += 1
    for t in mejores3: como_3ro[t] += 1
    combos["".join(sorted(POOL_DE[t] for t in mejores3))] += 1
    for t in POOL_DE: pts_tot[t] += st[t]["pts"]

out = pd.DataFrame([{"grupo": POOL_DE[t], "equipo": t, "1º": pos[t][0] / N_TORNEOS, "2º": pos[t][1] / N_TORNEOS,
                     "3º": pos[t][2] / N_TORNEOS, "4º": pos[t][3] / N_TORNEOS,
                     "avanza como 3º": como_3ro[t] / N_TORNEOS,
                     "avanza (total)": (pos[t][0] + pos[t][1]) / N_TORNEOS + como_3ro[t] / N_TORNEOS,
                     "pts medios": pts_tot[t] / N_TORNEOS} for t in POOL_DE]).sort_values(["grupo", "1º"], ascending=[True, False])
print(f"{N_TORNEOS:,} fases de grupos en {time.time()-t0:.0f}s\n")
print(out.round(3).to_string(index=False))
print("\nCombinaciones de grupos de los 4 mejores terceros (las más frecuentes):")
print(pd.Series(combos).sort_values(ascending=False).head(8).div(N_TORNEOS).round(3).to_string())
print(f"Combinaciones distintas vistas: {len(combos)} de 15")
print("\nControl: suma de 'avanza (total)' (debe dar 16):", round(out["avanza (total)"].sum(), 2))

10,000 fases de grupos en 152s

grupo equipo    1º    2º    3º    4º  avanza como 3º  avanza (total)  pts medios
    A    NZL 0.876 0.124 0.001 0.000           0.001           1.000      14.202
    A    AUS 0.124 0.860 0.015 0.001           0.012           0.996      10.667
    A    CHI 0.000 0.013 0.763 0.223           0.368           0.381       4.398
    A    HKG 0.000 0.003 0.221 0.776           0.083           0.086       1.779
    B    RSA 0.965 0.032 0.003 0.000           0.003           1.000      14.554
    B    GEO 0.021 0.502 0.453 0.024           0.410           0.934       8.083
    B    ITA 0.014 0.459 0.492 0.035           0.406           0.879       7.597
    B    ROU 0.000 0.006 0.052 0.941           0.027           0.034       1.053
    C    ARG 0.725 0.242 0.032 0.001           0.032           0.998      13.246
    C    FIJ 0.253 0.623 0.116 0.008           0.107           0.983      10.702
    C    ESP 0.021 0.128 0.735 0.115           0.573           0.723       5.

In [4]:
# =============== CELDA 4: CUADRO ELIMINATORIO Y CAMPEÓN (corre después de las celdas 1, 2 y 3) ===============
# Fuente de llaves y tabla de terceros: página del Mundial 2027 en Wikipedia (consistente con el artículo de World Rugby).
# Por cada combinación de 4 grupos con tercero clasificado: qué tercero enfrenta a A1, B1, C1, D1
TERCEROS = {
 "ABCD": "CDAB", "ABCE": "CEAB", "ABCF": "CFAB", "ABDE": "EDAB", "ABDF": "FDAB",
 "ABEF": "EFAB", "ACDE": "CDAE", "ACDF": "CDAF", "ACEF": "CEAF", "ADEF": "EDAF",
 "BCDE": "CDEB", "BCDF": "CDFB", "BCEF": "CEFB", "BDEF": "EDFB", "CDEF": "CDEF"}

# ---- probabilidad de que A le gane a B en un partido de eliminación (sin empates) ----
EQUIPOS = sorted(POOL_DE)
def p_gana(a, b):
    """Mundial, cancha neutral (Australia, de local), descanso igual. Se promedia en ambos sentidos para que sea simétrico."""
    def una(h, v):
        ventaja = HFA * (1 if h == "AUS" else -1 if v == "AUS" else 0)
        x = {"elo_diff": elo[h] + ventaja - elo[v], "neutral": int(ventaja == 0),
             "dif_forma_gan": ult.forma_gan[h] - ult.forma_gan[v], "dif_forma_margen": ult.forma_margen[h] - ult.forma_margen[v],
             "dif_descanso": 0, "comp_rwc": 1, "comp_rwc_qual": 0, "comp_six_nations": 0,
             "comp_rugby_championship": 0, "comp_tier2_comp": 0, "comp_warmup": 0}
        return x
    filas = pd.DataFrame([una(a, b), una(b, a)])[VARS]
    p = mod.predict_proba(filas)[:, 1]
    return (p[0] + 1 - p[1]) / 2
P = {(a, b): p_gana(a, b) for a in EQUIPOS for b in EQUIPOS if a != b}

def juega(a, b):
    return a if rng.random() < P[(a, b)] else b

def torneo():
    orden, mejores3, _ = fase_de_grupos()
    g1 = {p: l[0] for p, l in orden.items()}; g2 = {p: l[1] for p, l in orden.items()}
    clave = "".join(sorted(POOL_DE[t] for t in mejores3))
    t3 = {p: orden[p][2] for p in orden}
    vs_A, vs_B, vs_C, vs_D = (t3[g] for g in TERCEROS[clave])
    # octavos (numeración oficial)
    o = {1: juega(g2["C"], g2["F"]), 2: juega(g1["A"], vs_A), 3: juega(g1["E"], g2["D"]), 4: juega(g1["B"], vs_B),
         5: juega(g1["C"], vs_C),    6: juega(g1["D"], vs_D),  7: juega(g2["A"], g2["E"]), 8: juega(g1["F"], g2["B"])}
    octavos = set(g1.values()) | set(g2.values()) | set(t3[g] for g in "ABCDEF" if t3[g] in mejores3)
    cuartos = [o[i] for i in range(1, 9)]
    c = {1: juega(o[2], o[4]), 2: juega(o[1], o[3]), 3: juega(o[5], o[6]), 4: juega(o[7], o[8])}
    s1, s2 = juega(c[1], c[2]), juega(c[3], c[4])
    campeon = juega(s1, s2)
    return octavos, set(cuartos), set(c.values()), {s1, s2}, campeon

N_SIM = 10_000
cont = {t: np.zeros(5) for t in EQUIPOS}      # octavos, cuartos, semis, final, campeón
t0 = time.time()
for _ in range(N_SIM):
    octavos, cuartos, semis, final, campeon = torneo()
    for i, conj in enumerate((octavos, cuartos, semis, final)):
        for t in conj: cont[t][i] += 1
    cont[campeon][4] += 1

tabla_final = pd.DataFrame({t: v / N_SIM for t, v in cont.items()}, index=["octavos", "cuartos", "semis", "final", "campeón"]).T
tabla_final.insert(0, "grupo", [POOL_DE[t] for t in tabla_final.index])
tabla_final = tabla_final.sort_values("campeón", ascending=False)
print(f"{N_SIM:,} Mundiales simulados en {time.time()-t0:.0f}s\n")
print((tabla_final.assign(**{c: tabla_final[c].map("{:.1%}".format) for c in ["octavos", "cuartos", "semis", "final", "campeón"]})).to_string())
print("\nControl: suma de campeón =", round(tabla_final["campeón"].sum(), 3), "| suma de finalistas =", round(tabla_final["final"].sum(), 3))

10,000 Mundiales simulados en 144s

    grupo octavos cuartos  semis  final campeón
NZL     A  100.0%   98.9%  59.6%  48.8%   38.1%
RSA     B  100.0%   98.0%  50.1%  39.7%   28.5%
IRE     D   99.9%   82.4%  70.2%  36.2%   12.3%
FRA     E  100.0%   62.7%  54.3%  14.0%    6.8%
ENG     F  100.0%   89.3%  48.6%  24.9%    6.7%
AUS     A   99.6%   92.3%  41.9%  20.4%    4.5%
SCO     D   99.3%   49.6%  37.5%  10.1%    2.5%
ARG     C   99.7%   85.0%  21.2%   4.8%    0.4%
WAL     F   97.3%   42.6%   4.7%   0.2%    0.0%
FIJ     C   98.4%   57.2%   6.9%   0.5%    0.0%
GEO     B   93.7%   12.4%   2.6%   0.4%    0.0%
CAN     C    7.2%    0.1%   0.0%   0.0%    0.0%
ESP     C   72.7%    5.0%   0.1%   0.0%    0.0%
CHI     A   38.0%    2.7%   0.1%   0.0%    0.0%
ITA     B   87.6%    8.4%   1.4%   0.1%    0.0%
HKG     A    9.0%    0.4%   0.0%   0.0%    0.0%
ROU     B    3.5%    0.0%   0.0%   0.0%    0.0%
POR     D   40.2%    1.0%   0.1%   0.0%    0.0%
JPN     E   89.8%    6.6%   0.7%   0.0%    0.0%
SAM 